# V2 — Phase 1 — Identité INSEE période-aware

**Objectif.** Reconstruire `data-lake/clean/company_identity_periodic/` depuis le fichier SIRENE `StockUniteLegaleHistorique` afin de pouvoir joindre les features d'identité (NAF, forme juridique, tranche d'effectifs, statut administratif) à la **bonne date** plutôt qu'avec un snapshot terminal.

**Pourquoi.** En V1, la table `clean/company_identity` ne conservait qu'une ligne par SIREN (la plus récente). Le constructeur de features V1 a joint cette table sans filtre temporel, ce qui a contaminé les variables d'identité par des valeurs futures (cf. Phase 0 V1 : Run 1 AUC 0,87 illégitime, baissé à 0,76 après exclusion). En V1 ces variables ont été **exclues** ; en V2 nous les réintroduisons légitimement.

**Contrat de jointure aval.** Pour un couple `(siren, prediction_date)`, la ligne unique correspondante de la table V2 vérifie :

```
period_start <= prediction_date < period_end
```

La dernière période d'un SIREN reçoit `period_end = 9999-12-31` pour que l'inégalité reste valide.

**Délivrable.** Parquet `data-lake/clean/company_identity_periodic/company_identity_periodic.parquet` + `_manifest.json` + résultats d'audit collés dans `docs/v2/v2_phase_log.md`.

**Coût d'exécution.** Environ 5–10 minutes sur Colab CPU pour ~60 millions de lignes en entrée.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/zribi1/pfein.git'
BRANCH = 'ml-v2'
REPO_DIR = '/content/pfein'
BACKEND_DIR = f'{REPO_DIR}/back_end'

DRIVE_ROOT = '/content/drive/MyDrive/PFE ML Data/pfe_data'
DATA_LAKE = f'{DRIVE_ROOT}/data-lake'
DUCKDB_TMP = '/content/pfein_duckdb_tmp'

# V2 inputs / outputs
RAW_INSEE_HISTORIQUE = f'{DATA_LAKE}/raw/insee/bulk/stock_unite_legale_historique'
OUT_PERIODIC = f'{DATA_LAKE}/clean/company_identity_periodic'
OUT_PERIODIC_LOCAL = '/content/pfein_phase1_out'

# V1 reference (for comparison only, never written to)
V1_SNAPSHOT_IDENTITY = f'{DATA_LAKE}/clean/company_identity'

Path(DUCKDB_TMP).mkdir(parents=True, exist_ok=True)

print('BACKEND_DIR      =', BACKEND_DIR)
print('RAW (input)      =', RAW_INSEE_HISTORIQUE)
print('OUT (V2)         =', OUT_PERIODIC)
print('V1 snapshot      =', V1_SNAPSHOT_IDENTITY)

In [ ]:
import os

if not Path(REPO_DIR).exists():
    !git clone --branch "$BRANCH" "$REPO_URL" "$REPO_DIR"

%cd $REPO_DIR
!git fetch origin
!git switch "$BRANCH" || git switch -c "$BRANCH" "origin/$BRANCH"
!git pull --ff-only origin "$BRANCH"
%cd $BACKEND_DIR

os.environ['DUCKDB_TEMP_DIRECTORY'] = DUCKDB_TMP
!pip install -q -r collabs/requirements-colab.txt

## 2. Vérifier que la source brute SIRENE historique est présente

La table `StockUniteLegaleHistorique_utf8.parquet` doit avoir été ingérée par le pipeline INSEE bulk avant cette phase. Si elle est absente, l'ingestion INSEE doit être lancée — voir `docs/insee_report_section.md`.

In [ ]:
import duckdb

raw_files = list(Path(RAW_INSEE_HISTORIQUE).rglob('*.parquet'))
if not raw_files:
    raise SystemExit(
        f'Aucun parquet trouvé sous {RAW_INSEE_HISTORIQUE}.\n'
        f'Lancez d\'abord l\'ingestion INSEE bulk historique.'
    )
print(f'Fichiers parquet historique INSEE trouvés : {len(raw_files)}')
for f in raw_files[:5]:
    print(f'  - {f.relative_to(DATA_LAKE)} ({f.stat().st_size / 1e9:.2f} GB)')
if len(raw_files) > 5:
    print(f'  ... et {len(raw_files) - 5} autres.')

# Inspect schema
con = duckdb.connect()
raw_schema = con.execute(
    f"DESCRIBE SELECT * FROM read_parquet('{RAW_INSEE_HISTORIQUE}/**/*.parquet', union_by_name=true) LIMIT 0"
).df()
raw_row_count = con.execute(
    f"SELECT COUNT(*) FROM read_parquet('{RAW_INSEE_HISTORIQUE}/**/*.parquet', union_by_name=true)"
).fetchone()[0]
con.close()

print(f'\nLignes brutes (toutes périodes confondues) : {raw_row_count:,}')
print(f'Colonnes ({len(raw_schema)}) :')
print(raw_schema['column_name'].tolist())

## 3. Lancer le build de la table périodique

Le script `app.tools.v2.build_company_identity_periodic` lit le raw, normalise les noms de colonnes, dérive `period_end` (début de la période suivante par SIREN) et écrit `company_identity_periodic.parquet` avec un `_manifest.json` d'audit.

In [ ]:
import shlex, subprocess, sys, time

cmd = [
    sys.executable, '-u',
    '-m', 'app.tools.v2.build_company_identity_periodic',
    '--raw-dir', RAW_INSEE_HISTORIQUE,
    '--out-dir', OUT_PERIODIC_LOCAL,
    '--duckdb-temp-dir', DUCKDB_TMP,
    '--overwrite',
]
print(' '.join(shlex.quote(p) for p in cmd))
start = time.time()
result = subprocess.run(cmd, capture_output=True, text=True)
elapsed = time.time() - start
print(result.stdout)
if result.returncode != 0:
    print('STDERR:')
    print(result.stderr)
    raise SystemExit(f'Build failed with code {result.returncode}')
print(f'\nDurée du build : {elapsed/60:.1f} min')

In [ ]:
# Move the local staged parquet + manifest to the Drive output dir.
# Doing this after the build avoids Drive FUSE page-cache OOM during COPY.
import shutil, json
src_dir = Path(OUT_PERIODIC_LOCAL)
dst_dir = Path(OUT_PERIODIC)
dst_dir.mkdir(parents=True, exist_ok=True)
for f in src_dir.iterdir():
    target = dst_dir / f.name
    if target.exists():
        target.unlink()
    shutil.move(str(f), str(target))
    print(f'Moved {f.name} -> {target}')
shutil.rmtree(src_dir, ignore_errors=True)

# Rewrite the manifest so out_path points at the Drive location.
# Downstream cells read manifest['out_path'] for audit + spot-checks.
manifest_path = dst_dir / '_manifest.json'
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
manifest['out_path'] = str(dst_dir / 'company_identity_periodic.parquet')
manifest['staged_from'] = str(src_dir)
manifest_path.write_text(json.dumps(manifest, indent=2, default=str), encoding='utf-8')

print('
Sync done. Drive output:')
for f in dst_dir.iterdir():
    print(f'  {f.name}  ({f.stat().st_size / 1e9:.2f} GB)')
print(f"
manifest['out_path'] = {manifest['out_path']}")


## 4. Audit des résultats

In [ ]:
import json

manifest_path = Path(OUT_PERIODIC) / '_manifest.json'
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))

print('=== AUDIT DE LA TABLE PERIODIQUE ===')
for k in [
    'total_rows', 'unique_sirens', 'rows_per_siren',
    'latest_period_rows', 'min_period_start', 'max_period_start',
    'avg_period_years',
]:
    print(f'  {k:30s} = {manifest.get(k)}')

print('\n=== INTEGRITE TEMPORELLE ===')
for k in ['rows_with_overlap', 'overlap_rate', 'rows_with_gap', 'gap_rate']:
    print(f'  {k:30s} = {manifest.get(k)}')

# Validation thresholds (cf. v2_roadmap.md Phase 1)
assert manifest['overlap_rate'] < 0.001, (
    f'Taux d\'overlap trop élevé : {manifest["overlap_rate"]}. '
    f'Investiguer avant de passer à la Phase 2.'
)
print('\n✅ Critères d\'intégrité respectés (overlap < 0.1%).')

## 5. Spot-check sur 10 SIRENs aléatoires multi-périodes

Vérifier visuellement que pour des entreprises ayant plusieurs périodes, on retrouve bien le changement d'attribut (NAF, forme juridique, statut administratif) attendu à la date appropriée.

In [ ]:
import pandas as pd

out_path = manifest['out_path']
con = duckdb.connect()

# Pick 10 random SIRENs that have at least 3 periods (interesting cases)
sample_sirens = con.execute(f"""
    SELECT siren
    FROM (
        SELECT siren, COUNT(*) AS n
        FROM read_parquet('{out_path}')
        GROUP BY siren
        HAVING COUNT(*) >= 3
    )
    USING SAMPLE 10 ROWS (RESERVOIR, 42)
""").df()

print(f'10 SIRENs multi-périodes tirés au hasard (graine 42) :')
print(sample_sirens['siren'].tolist())

for siren in sample_sirens['siren'].tolist()[:5]:
    print(f'\n--- SIREN {siren} ---')
    df_s = con.execute(f"""
        SELECT period_start, period_end,
               activity_code, legal_category_code,
               employee_size_bracket, administrative_status,
               is_latest_period
        FROM read_parquet('{out_path}')
        WHERE siren = '{siren}'
        ORDER BY period_start
    """).df()
    print(df_s.to_string(index=False))

con.close()

## 6. Comparaison avec le snapshot V1 (`clean/company_identity`)

Pour valider que V2 contient bien plus de lignes et que la jointure sera correcte aux dates passées.

In [ ]:
v1_path = Path(V1_SNAPSHOT_IDENTITY)
if not any(v1_path.rglob('*.parquet')):
    print('V1 snapshot absent — comparaison ignorée.')
else:
    con = duckdb.connect()
    v1_stats = con.execute(f"""
        SELECT
            COUNT(*) AS rows,
            COUNT(DISTINCT siren) AS sirens
        FROM read_parquet('{V1_SNAPSHOT_IDENTITY}/**/*.parquet', union_by_name=true)
    """).fetchone()
    con.close()

    print('V1 snapshot vs V2 périodique :')
    print(f'  V1 rows                = {v1_stats[0]:,}')
    print(f'  V1 sirens              = {v1_stats[1]:,}')
    print(f'  V2 rows                = {manifest["total_rows"]:,}')
    print(f'  V2 sirens              = {manifest["unique_sirens"]:,}')
    if v1_stats[0]:
        print(f'  Multiplicateur lignes  = {manifest["total_rows"] / v1_stats[0]:.2f}×')
    print(f'  Périodes par SIREN     = {manifest["rows_per_siren"]:.2f}')

## 7. Test de jointure temporelle simulée

Pour 100 SIRENs aléatoires, on simule une jointure pour `prediction_date = 2020-12-31` et `prediction_date = 2023-12-31` et on vérifie que chaque couple `(siren, date)` matche **exactement une** période.

In [ ]:
con = duckdb.connect()
for test_date in ['2020-12-31', '2023-12-31']:
    res = con.execute(f"""
        WITH sample AS (
            SELECT DISTINCT siren
            FROM read_parquet('{out_path}')
            USING SAMPLE 100 ROWS (RESERVOIR, 42)
        ),
        joined AS (
            SELECT s.siren, COUNT(p.siren) AS n_matches
            FROM sample s
            LEFT JOIN read_parquet('{out_path}') p
                ON p.siren = s.siren
               AND p.period_start <= DATE '{test_date}'
               AND p.period_end   >  DATE '{test_date}'
            GROUP BY s.siren
        )
        SELECT
            COUNT(*) AS total_sirens,
            SUM(CASE WHEN n_matches = 0 THEN 1 ELSE 0 END) AS no_match,
            SUM(CASE WHEN n_matches = 1 THEN 1 ELSE 0 END) AS one_match,
            SUM(CASE WHEN n_matches > 1 THEN 1 ELSE 0 END) AS multiple_matches
        FROM joined
    """).fetchone()
    print(f'Jointure temporelle à {test_date} sur 100 SIRENs aléatoires :')
    print(f'  total                  = {res[0]}')
    print(f'  aucun match            = {res[1]} (entreprises créées après {test_date})')
    print(f'  1 match exactement     = {res[2]}  ← attendu')
    print(f'  matches multiples      = {res[3]}  ← doit être 0')
    if res[3] > 0:
        print('  ⚠️  ANOMALIE : des SIRENs matchent plusieurs périodes pour une même date. Investiguer.')
    print()
con.close()

## 8. Reporter les résultats dans le journal V2

Copier la sortie ci-dessous dans `docs/v2/v2_phase_log.md` section Phase 1.

In [ ]:
from datetime import datetime

log_entry = f"""## Phase 1 — Identité INSEE période-aware

**Date d'exécution :** {datetime.now().strftime('%Y-%m-%d')}
**Notebook :** `collabs/v2/v2_phase_1_identity_periodic.ipynb`
**Script :** `app/tools/v2/build_company_identity_periodic.py`

### Résultats

- **Lignes en entrée** : {raw_row_count:,}
- **Lignes en sortie** : {manifest['total_rows']:,}
- **SIRENs uniques** : {manifest['unique_sirens']:,}
- **Périodes par SIREN** : {manifest['rows_per_siren']:.2f}
- **Période min** : {manifest['min_period_start']}
- **Période max** : {manifest['max_period_start']}
- **Lignes avec overlap** : {manifest['rows_with_overlap']} ({manifest['overlap_rate']*100:.4f}%)
- **Lignes avec gap** : {manifest['rows_with_gap']} ({manifest['gap_rate']*100:.4f}%)

### Audit

- ✅ Overlap rate < 0.1% (critère d'intégrité respecté)
- ✅ Jointure temporelle vérifiée sur 100 SIRENs aléatoires aux dates 2020-12-31 et 2023-12-31
- Test multi-période spot-check : 10 SIRENs vérifiés visuellement

### Artefacts produits

- `data-lake/clean/company_identity_periodic/company_identity_periodic.parquet`
- `data-lake/clean/company_identity_periodic/_manifest.json`

### Décision

**Passer à la Phase 2** (build des features V2) une fois la sortie validée par lecture du tableau ci-dessus.
"""

print(log_entry)